# Structured Output 
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [15]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model('groq:openai/gpt-oss-120b')

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x113a77610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113b08050>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")

In [17]:
model_with_structure = model.with_structured_output(Movie)

In [18]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x113a77610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113b08050>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the mov

In [19]:
model.invoke("Provide details about the movie 'Inception'")

AIMessage(content='**Inception (2010) – Quick Reference**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Year** | 2010 |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Running Time** | 148 minutes |\n| **Country** | United States |\n| **Language** | Primarily English (some French, Japanese, and Russian dialogue) |\n| **Budget** | Approx. **$160\u202fmillion** |\n| **Box‑Office Gross** | ~**$836\u202fmillion** worldwide |\n| **MPAA Rating** | PG‑13 (Violence, Some Language, Brief Sexual Content) |\n| **Director / Writer** | Christopher Nolan |\n| **Producers** | Emma Thomas, Christopher Nolan, Jordan Goldberg (executive) |\n| **Cinematography** | Wally Pfister |\n| **Music** | Hans Zimmer |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Distributor** | Warner Bros. Pictures |\n\n---\n\n### Main Cast & Characters\n| Actor/Actress | Character | Brief Note |\n|---------------|-----------|----

In [20]:
model_with_structure.invoke("Provide details about the movie 'Inception'")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output alongside parsed structure

In [21]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="The year the movie was released")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The movies rating out of 10")
model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie 'Inception'")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie \'Inception\'. We have a function Movie that likely returns details given director, rating, title, year. We need to provide details. We need to call the function with appropriate parameters. The function signature: Movie takes an object with director, rating, title, year. We need to fill these. Inception: director Christopher Nolan, rating maybe 8.8 (IMDb) out of 10, title "Inception", year 2010. Use function.', 'tool_calls': [{'id': 'fc_9576f405-2646-4732-b379-c0af01c7a2e4', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 160, 'total_tokens': 315, 'completion_time': 0.327724305, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.063410566, 'prompt_tokens_details': None, 'queue_tim

### Nested Structure

In [22]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str 
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(description="The budget of the movie in millions of dollars")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie 'Inception'")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Prof. Stephen Miles'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Tom Berenger', role='Peter Browning')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypeDict
TypeDict provides a simple alternative using Python's built-in typing, ideal when you don't need runtime validation

In [24]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ...,"The title of the movie"]
    year: Annotated[int, ...,"The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movies rating out of 10"]

model_with_typeddict_structure = model.with_structured_output(MovieDict)
response = model_with_typeddict_structure.invoke("Provide details about the movie 'Interstellar'")
response


{'director': 'Christopher Nolan',
 'rating': 8.6,
 'title': 'Interstellar',
 'year': 2014}

In [26]:
class Actor(TypedDict):
    name:str 
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(description="Budget in millions USD")
    

model_with_typeddict_structure = model.with_structured_output(MovieDetails)
response = model_with_typeddict_structure.invoke("Provide details about the movie 'Inception'")
response


{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Marion Cotillard', 'role': 'Mal'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'},
  {'name': 'Dileep Rao', 'role': 'Yusuf'},
  {'name': 'Pete Postlethwaite', 'role': 'Mr. Browning'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [27]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Data Classes
A data class is a class typically containing mainly data, although there is not really any restrictions. You create it using the @dataclass decorator.

In [ ]:
import os 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [31]:
from unittest import result
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name:str = Field(...,description="The name of the person")
    email:str = Field(...,description="The email of the person")
    phone:str = Field(...,description="The phone number of the person")


agent = create_agent(model ="groq:openai/gpt-oss-120b", response_format=ContactInfo)

result = agent.invoke({
    "messages":[
        {"role":"user", "content":"Extract the contact information of the person from the following text: John Doe, john.doe@example.com, 123-456-7890"}
    ]
})

print(result)

print(result['structured_response'])



{'messages': [HumanMessage(content='Extract the contact information of the person from the following text: John Doe, john.doe@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='08bcf9c1-e689-4ea3-8166-47c2053bb6b2'), AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from text. We must output only JSON object conforming to ContactInfo schema: name, email, phone. Must be compact JSON, parse as JSON, include all fields. So produce:\n\n{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}\n\nWe must output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 245, 'total_tokens': 356, 'completion_time': 0.239821943, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.012817427, 'prompt_tokens_details': None, 'queue_time': 0.250978198, 'total_time': 0.25263937}, 'mod

### Data class

In [32]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of a person"""
    name:str
    email:str
    phone:str

agent = create_agent(model ="groq:openai/gpt-oss-120b", response_format=ContactInfo)

result = agent.invoke({
    "messages":[
        {"role":"user", "content":"Extract the contact information of the person from the following text: John Doe, john.doe@example.com, 123-456-7890"}
    ]
})

print(result)

print(result['structured_response'])


{'messages': [HumanMessage(content='Extract the contact information of the person from the following text: John Doe, john.doe@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='ca6ff36d-09d8-473f-808b-22335d5582f0'), AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={'reasoning_content': 'The user asks: "Extract the contact information of the person from the following text: John Doe, john.doe@example.com, 123-456-7890". The response format is a JSON object adhering to the ContactInfo schema: fields name, email, phone all required.\n\nWe must output ONLY the final JSON object, compact (no whitespace). So produce something like:\n\n{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}\n\nMake sure JSON is valid.'}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 220, 'total_tokens': 362, 'completion_time': 0.301769925, 'completion_tokens_details': {'re